# Phase 25 — MLP & LightGBM

Δοκιμή δύο ακόμα classifiers πάνω σε TF-IDF features:
- **MLP** (Multi-Layer Perceptron) — neural network χωρίς transformers
- **LightGBM** — gradient boosting (πιο γρήγορο από XGBoost)

**Baseline για σύγκριση:**
- Naive Bayes:  0.3122
- Random Forest: 0.4775
- Logistic Regression: 0.6523
- SVM (tuned): 0.7419 ← best classical

In [1]:
!pip install lightgbm -q

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


In [3]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:550].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:550].apply(preprocess_text)
test['combined']  = test['title'].apply(preprocess_text)  + ' ' + test['text'].str[:550].apply(preprocess_text)

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)
X_train = tfidf.fit_transform(train['combined'])
X_valid = tfidf.transform(valid['combined'])
X_test  = tfidf.transform(test['combined'])

y_train_hazard  = train['hazard-category']
y_train_product = train['product-category']
y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

# Label encoding για LightGBM
le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(pd.concat([train['hazard-category'],  valid['hazard-category']]))
le_product.fit(pd.concat([train['product-category'], valid['product-category']]))

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')
print(f'TF-IDF features: {X_train.shape[1]}')

Train: 5082 | Valid: 565 | Test: 997
TF-IDF features: 50000


In [4]:
def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    mask = (y_true_hazard == y_pred_hazard)
    f1_product = f1_score(
        y_true_product[mask], y_pred_product[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'  Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

## 1. MLP (Multi-Layer Perceptron)

Neural network με 2 hidden layers πάνω σε TF-IDF features.
Διαφορά από BERT: δεν έχει attention mechanism και δεν καταλαβαίνει context.

In [7]:
# Encode labels σε αριθμούς
le_h = LabelEncoder()
le_p = LabelEncoder()
y_train_h_enc = le_h.fit_transform(y_train_hazard)
y_train_p_enc = le_p.fit_transform(y_train_product)

print('=== MLP — HAZARD ===')
mlp_hazard = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    max_iter=50,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=False
)
mlp_hazard.fit(X_train, y_train_h_enc)
print(f'Iterations: {mlp_hazard.n_iter_}')

print('\n=== MLP — PRODUCT ===')
mlp_product = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    max_iter=50,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=False
)
mlp_product.fit(X_train, y_train_p_enc)
print(f'Iterations: {mlp_product.n_iter_}')

pred_hazard_mlp  = le_h.inverse_transform(mlp_hazard.predict(X_valid))
pred_product_mlp = le_p.inverse_transform(mlp_product.predict(X_valid))

print('\n=== ΑΞΙΟΛΟΓΗΣΗ MLP ===')
score_mlp = official_st1_score(
    y_valid_hazard, pred_hazard_mlp,
    y_valid_product, pred_product_mlp
)

pd.DataFrame({
    'id': test['id'],
    'hazard-category':  le_h.inverse_transform(mlp_hazard.predict(X_test)),
    'product-category': le_p.inverse_transform(mlp_product.predict(X_test))
}).to_csv('submission_mlp.csv', index=False)
print('\nΑποθηκεύτηκε: submission_mlp.csv')

=== MLP — HAZARD ===
Iterations: 18

=== MLP — PRODUCT ===
Iterations: 11

=== ΑΞΙΟΛΟΓΗΣΗ MLP ===
  macro-F1 Hazard:                    0.7773
  Σωστά hazard predictions:           519/565 (91.9%)
  macro-F1 Product (given correct h): 0.6657
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.7215

Αποθηκεύτηκε: submission_mlp.csv


## 2. LightGBM (Gradient Boosting)

Gradient boosting με decision trees — πολύ πιο γρήγορο από XGBoost.
Χρειάζεται dense matrix αντί sparse, οπότε μειώνουμε τα features σε 5000.


In [ ]:
# LightGBM δουλεύει καλύτερα με λιγότερα features
# Χρησιμοποιούμε 5000 αντί 50000
tfidf_lgb = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)
X_train_lgb = tfidf_lgb.fit_transform(train['combined']).toarray()
X_valid_lgb = tfidf_lgb.transform(valid['combined']).toarray()
X_test_lgb  = tfidf_lgb.transform(test['combined']).toarray()

# Label encoding
y_train_hazard_enc  = le_hazard.transform(y_train_hazard)
y_train_product_enc = le_product.transform(y_train_product)

print(f'LightGBM TF-IDF features: {X_train_lgb.shape[1]}')

print('\n=== LightGBM — HAZARD ===')
lgb_hazard = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)
lgb_hazard.fit(X_train_lgb, y_train_hazard_enc)

print('=== LightGBM — PRODUCT ===')
lgb_product = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)
lgb_product.fit(X_train_lgb, y_train_product_enc)

pred_hazard_lgb  = le_hazard.inverse_transform(lgb_hazard.predict(X_valid_lgb))
pred_product_lgb = le_product.inverse_transform(lgb_product.predict(X_valid_lgb))

print('\n=== ΑΞΙΟΛΟΓΗΣΗ LightGBM ===')
score_lgb = official_st1_score(
    y_valid_hazard, pred_hazard_lgb,
    y_valid_product, pred_product_lgb
)

# Submission
pd.DataFrame({
    'id': test['id'],
    'hazard-category': le_hazard.inverse_transform(lgb_hazard.predict(X_test_lgb)),
    'product-category': le_product.inverse_transform(lgb_product.predict(X_test_lgb))
}).to_csv('submission_lgb.csv', index=False)
print('\nΑποθηκεύτηκε: submission_lgb.csv')

## 3. Σύγκριση Όλων των Classical Μεθόδων

In [ ]:
print('='*55)
print('ΣΥΓΚΡΙΣΗ CLASSICAL ΜΕΘΟΔΩΝ (validation ST1 score)')
print('='*55)

results = [
    ('Naive Bayes',          0.3122),
    ('Random Forest',        0.4775),
    ('Word2Vec + SVM',       0.4629),
    ('GloVe + SVM',          0.4618),
    ('FastText + SVM',       0.4716),
    ('Logistic Regression',  0.6523),
    ('MLP',                  score_mlp),
    ('LightGBM',             score_lgb),
    ('SVM (tuned)',          0.7419),
]
results.sort(key=lambda x: x[1])

for name, score in results:
    bar = '█' * int(score * 40)
    print(f'{name:25s} {score:.4f}  {bar}')

print('\n--- Transformers (για σύγκριση) ---')
print(f'{"DistilBERT":25s} 0.7606')
print(f'{"BERT-base + Focal":25s} 0.8040')
print(f'{"Best Ensemble":25s} 0.8173')